In [16]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# Load the dataset
file_path = "german_data_main.xlsx"
df = pd.read_excel(file_path)

# Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=['object']).columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Handle missing values
num_imputer = SimpleImputer(strategy='mean')
df[numerical_cols] = num_imputer.fit_transform(df[numerical_cols])

cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

# Encode categorical variables using LabelEncoder
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Scale numerical variables using MinMaxScaler
scaler = MinMaxScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# Save the processed dataset
processed_file_path = "german_data_preprocessed.xlsx"
df.to_excel(processed_file_path, index=False)

print(f"Preprocessed data saved to {processed_file_path}")


Preprocessed data saved to german_data_preprocessed.xlsx


In [17]:
# After preprocessing, perform additional checks

# 1. Check for missing values
missing_values = df.isnull().sum()
print("Missing Values in Each Column:")
print(missing_values)

# 2. Verify that numerical columns are within the range [0, 1]
print("\nNumerical Columns Summary (should be in range [0,1]):")
print(df[numerical_cols].describe())

# 3. Check unique values for categorical columns
print("\nUnique values in categorical columns (after Label Encoding):")
for col in categorical_cols:
    unique_vals = df[col].unique()
    print(f"{col}: {unique_vals}")

# 4. Check data types
print("\nData Types of All Columns:")
print(df.dtypes)

# Optionally, save a sample of the data to visually inspect it
df.sample(5).to_excel("german_data_preprocessed_sample.xlsx", index=False)
print("\nA sample of the preprocessed data has been saved to 'german_data_preprocessed_sample.xlsx'")


Missing Values in Each Column:
Status_of_existing_checking_account    0
Duration_in_month                      0
Credit_history                         0
Purpose                                0
Credit_amount                          0
Savings_account_bonds                  0
Present_employment_since               0
Installment_rate                       0
Personal_status_and_sex                0
Other_debtors_guarantors               0
Present_residence_since                0
Age_in_years                           0
Other_installment_plans                0
Housing                                0
Number_of_existing_credits             0
Job                                    0
Number_of_people_liable                0
Credit_risk                            0
dtype: int64

Numerical Columns Summary (should be in range [0,1]):
       Duration_in_month  Credit_amount  Installment_rate  \
count        5000.000000    5000.000000       5000.000000   
mean            0.251281       0.167562  

In [11]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, Embedding
from tensorflow.keras.layers import MultiHeadAttention, GlobalAveragePooling1D
from sklearn.model_selection import train_test_split

# Load the preprocessed dataset
file_path = "german_data_preprocessed.xlsx"
df = pd.read_excel(file_path)

# Split features and target (assuming the last column is the target)
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# Reshape input data for Transformer (samples, time steps, features)
X = X.reshape((X.shape[0], 1, X.shape[1]))  # As a sequence with a single time step

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# Transformer Model components
def transformer_encoder(inputs, head_size, num_heads, ff_size, dropout_rate=0.1):
    # Multi-Head Attention Layer
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(inputs, inputs)
    attention = Dropout(dropout_rate)(attention)
    attention = LayerNormalization()(inputs + attention)
    
    # Feed-Forward Layer
    ff = Dense(ff_size, activation='relu')(attention)
    ff = Dropout(dropout_rate)(ff)
    ff = Dense(inputs.shape[-1])(ff)
    ff = Dropout(dropout_rate)(ff)
    output = LayerNormalization()(attention + ff)
    
    return output

# Build the Transformer model
input_layer = Input(shape=(X_train.shape[1], X_train.shape[2]))

# Apply Transformer Encoder
x = transformer_encoder(input_layer, head_size=64, num_heads=4, ff_size=128, dropout_rate=0.2)

# Global Average Pooling
x = GlobalAveragePooling1D()(x)

# Fully Connected Layers
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(32, activation='relu')(x)
x = Dropout(0.3)(x)

# Output Layer
output_layer = Dense(1, activation='sigmoid')(x)

# Create and compile the model
model = Model(inputs=input_layer, outputs=output_layer)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Summary of the model
model.summary()

# Train the model
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test))

# Evaluate the model
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)    │ (None, 1, 17)             │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ multi_head_attention_12       │ (None, 1, 17)             │          18,193 │ input_layer_8[0][0],       │
│ (MultiHeadAttention)          │                           │                 │ input_layer_8[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_66 (Dropout)          │ (None, 1, 17)             │               0 │ multi_head_attention_12[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_24 (Add)                  │ (None, 1, 17)             │               0 │ input_layer_8[0][0],       │
│                               │                           │                 │ dropout_66[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization_27        │ (None, 1, 17)             │              34 │ add_24[0][0]               │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_49 (Dense)              │ (None, 1, 128)            │           2,304 │ layer_normalization_27[0]… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_67 (Dropout)          │ (None, 1, 128)            │               0 │ dense_49[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_50 (Dense)              │ (None, 1, 17)             │           2,193 │ dropout_67[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_68 (Dropout)          │ (None, 1, 17)             │               0 │ dense_50[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_25 (Add)                  │ (None, 1, 17)             │               0 │ layer_normalization_27[0]… │
│                               │                           │                 │ dropout_68[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization_28        │ (None, 1, 17)             │              34 │ add_25[0][0]               │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling1d_8    │ (None, 17)                │               0 │ layer_normalization_28[0]… │
│ (GlobalAveragePooling1D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_51 (Dense)              │ (None, 64)                │           1,152 │ global_average_pooling1d_… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_69 (Dropout)          │ (None, 64)                │               0 │ dense_51[0][0]             │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 26,023 (101.65 KB)

 Trainable params: 26,023 (101.65 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.6510 - loss: 0.6448 - val_accuracy: 0.7240 - val_loss: 0.5586
Epoch 2/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7023 - loss: 0.5819 - val_accuracy: 0.7300 - val_loss: 0.5355
Epoch 3/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6961 - loss: 0.5669 - val_accuracy: 0.7400 - val_loss: 0.5384
Epoch 4/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7168 - loss: 0.5445 - val_accuracy: 0.7280 - val_loss: 0.5124
Epoch 5/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7353 - loss: 0.5248 - val_accuracy: 0.7380 - val_loss: 0.4991
Epoch 6/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7286 - loss: 0.5235 - val_accuracy: 0.7400 - val_loss: 0.4894
Epoch 7/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7417 - loss: 0.5060 - val_accuracy: 0.7480 - val_loss: 0.4813
Epoch 8/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7421 - loss: 0.5077 - val_accu

In [19]:
# Save the trained Keras model
# Save the model in the .keras format
model.save("german_credit_model.keras")
print("Model saved as 'german_credit_model.keras'")

# Optionally, save the preprocessing objects using joblib or pickle
import joblib

joblib.dump(scaler, "minmax_scaler.pkl")
joblib.dump(label_encoders, "label_encoders.pkl")
print("Preprocessing objects saved as minmax_scaler.pkl and label_encoders.pkl")


Model saved as 'german_credit_model.keras'
Preprocessing objects saved as minmax_scaler.pkl and label_encoders.pkl
